# Data Cleaning and Text Preprocessing

This notebook performs the complete preparation pipeline in one place:

1. Load the merged review dataset.
2. Apply general data cleaning with `clean.py`.
3. Create classical and transformer text features with `text_preprocessing.py`.
4. Validate the final dataset.
5. Save one processed CSV file.

The final output is:

```text
data/processed/processed_reviews.csv
```

## Downstream usage

For classical machine-learning models:

```python
X = df_processed["classical_text"]
y = df_processed["rating"]
```

For transformer models:

```python
X = df_processed["transformer_text"]
y = df_processed["rating"]
```

Transformer tokenization, padding, truncation, and attention-mask creation should happen later in the transformer modeling notebook.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

from src.voxforge.data.text_preprocessing import (
    create_model_text_features,
    ensure_nltk_resources,
    validate_model_text_features,
)
from src.voxforge.data.clean import clean_dataset, validate_clean_dataset
from voxforge.data.config import INTERIM_DATA_DIR, PROCESSED_DATA_DIR


## Locate the project root

This allows the notebook to work when it is executed from the `notebooks` directory.

In [ ]:
# locate the project root
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [ ]:
# load the interim merged dataset
merged_path = INTERIM_DATA_DIR / "merged_reviews.csv"

merged_df = pd.read_csv(
    merged_path,
    low_memory=False,
)
print(f"Raw dataset shape: {merged_df.shape}")
display(merged_df.head())

## Apply general data cleaning

The original DataFrame is preserved. Cleaning is applied to a copy.

In [ ]:
df_clean = clean_dataset(merged_df.copy())

validate_clean_dataset(df_clean)

print(f"Cleaned dataset shape: {df_clean.shape}")
print("Cleaning validation passed.")

display(df_clean.head())

## Create model-specific text features

The preprocessing pipeline creates:

- `normalized_combined_text`
- `classical_text`
- `transformer_text`

Normalized duplicate removal happens before the classical and transformer branches, which keeps both columns aligned.

In [ ]:
ensure_nltk_resources()

df_processed = create_model_text_features(
    df_clean.copy(),
    remove_duplicates=True,
    include_rating_in_duplicate_key=True,
)

validate_model_text_features(df_processed)

print(f"Processed dataset shape: {df_processed.shape}")
print("Text preprocessing validation passed.")

## Preview the final text columns

In [ ]:
preview_columns = [
    "review_title",
    "review_text",
    "normalized_combined_text",
    "classical_text",
    "transformer_text",
    "rating",
]

display(df_processed[preview_columns].head())

## Final quality checks

In [ ]:
quality_summary = pd.Series(
    {
        "raw_rows": len(merged_df),
        "cleaned_rows": len(df_clean),
        "processed_rows": len(df_processed),
        "removed_during_cleaning": len(merged_df) - len(df_clean),
        "removed_during_normalized_deduplication": len(df_clean) - len(df_processed),
        "missing_review_text": int(df_processed["review_text"].isna().sum()),
        "invalid_ratings": int(
            (~df_processed["rating"].between(1, 5, inclusive="both")).sum()
        ),
        "empty_transformer_text": int(
            df_processed["transformer_text"]
            .astype("string")
            .str.len()
            .eq(0)
            .sum()
        ),
        "empty_classical_text": int(
            df_processed["classical_text"]
            .astype("string")
            .str.len()
            .eq(0)
            .sum()
        ),
    },
    name="quality_summary",
)

quality_summary

## Save one processed CSV file

In [ ]:
processed_path = PROCESSED_DATA_DIR / "processed_reviews.csv"

df_processed.to_csv(processed_path, index=False)

print(f"Saved processed dataset to: {processed_path}")

## Final verification

In [ ]:
assert processed_path.exists()
assert "classical_text" in df_processed.columns
assert "transformer_text" in df_processed.columns
assert len(df_processed["classical_text"]) == len(df_processed["transformer_text"])

print("Final dataset is ready for modeling.")